In [11]:
# Install Detectron2
!pip install 'git+https://github.com/facebookresearch/detectron2.git@v0.6' -q


  Preparing metadata (setup.py) ... done


In [14]:
# FIX Pillow compatibility for Detectron2
from PIL import Image

if not hasattr(Image, "LINEAR"):
    Image.LINEAR = Image.BILINEAR

In [15]:
import os
import json
import random
import shutil
from sklearn.model_selection import train_test_split

from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import load_coco_json, register_coco_instances
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo

import cv2

DATASET_ROOT = "/kaggle/input/datasets/sonlest/bom-dataset-v3/BOM-Dataset-v3"
ANNOTATION_FILE = os.path.join(DATASET_ROOT, "annotations/annotations_v1.json")
IMAGE_DIR = os.path.join(DATASET_ROOT, "images")

OUTPUT_DIR = "/kaggle/working/detectron2_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ImportError: cannot import name 'is_directory' from 'PIL._util' (/usr/local/lib/python3.12/dist-packages/PIL/_util.py)

In [ ]:
def split_dataset():
    print("🔀 Splitting dataset...")

    with open(ANNOTATION_FILE) as f:
        coco = json.load(f)

    images = coco["images"]

    train_imgs, temp_imgs = train_test_split(images, test_size=0.4, random_state=42)
    val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.25, random_state=42)

    print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}")

    return coco, train_imgs, val_imgs, test_imgs

In [ ]:
def create_subset(coco, selected_images, name):
    image_ids = set(img["id"] for img in selected_images)

    annotations = [ann for ann in coco["annotations"] if ann["image_id"] in image_ids]

    subset = {
        "images": selected_images,
        "annotations": annotations,
        "categories": coco["categories"]
    }

    save_path = f"/kaggle/working/{name}.json"
    with open(save_path, "w") as f:
        json.dump(subset, f)

    return save_path

In [ ]:
def register_dataset(train_json, val_json):
    print("📚 Registering dataset...")

    register_coco_instances("bom_train", {}, train_json, IMAGE_DIR)
    register_coco_instances("bom_val", {}, val_json, IMAGE_DIR)

In [ ]:
def setup_config():
    print("⚙️ Setting up config...")

    cfg = get_cfg()
    cfg.merge_from_file(
        model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")
    )

    cfg.DATASETS.TRAIN = ("bom_train",)
    cfg.DATASETS.TEST = ("bom_val",)

    cfg.DATALOADER.NUM_WORKERS = 2

    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
        "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"
    )

    cfg.SOLVER.IMS_PER_BATCH = 2
    cfg.SOLVER.BASE_LR = 0.00025   # giảm vì dataset nhỏ
    cfg.SOLVER.MAX_ITER = 1000     # có thể tăng sau

    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 3

    cfg.OUTPUT_DIR = OUTPUT_DIR

    return cfg

In [ ]:
def train_model(cfg):
    print("🚀 Training model...")

    trainer = DefaultTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()

    return cfg

In [ ]:
def load_predictor(cfg):
    cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

    predictor = DefaultPredictor(cfg)
    return predictor

In [ ]:
def process_image(image_path, predictor, metadata):
    image = cv2.imread(image_path)
    outputs = predictor(image)

    instances = outputs["instances"].to("cpu")

    boxes = instances.pred_boxes.tensor.numpy()
    scores = instances.scores.numpy()
    classes = instances.pred_classes.numpy()

    results = []

    for i, (box, score, cls) in enumerate(zip(boxes, scores, classes)):
        x1, y1, x2, y2 = map(int, box)

        crop = image[y1:y2, x1:x2]

        crop_path = image_path.replace(".png", f"_crop_{i}.png")
        cv2.imwrite(crop_path, crop)

        obj = {
            "id": i,
            "class": metadata.thing_classes[cls],
            "confidence": float(score),
            "bbox": {
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2
            },
            "ocr_content": ""  # placeholder OCR
        }

        results.append(obj)

    return results

In [ ]:
def run_ocr_stub(objects):
    for obj in objects:
        if obj["class"] in ["Note", "Table"]:
            obj["ocr_content"] = "OCR_PLACEHOLDER"
    return objects

In [ ]:
def save_output(image_path, objects):
    output = {
        "image": os.path.basename(image_path),
        "objects": objects
    }

    save_path = image_path.replace(".png", ".json")

    with open(save_path, "w") as f:
        json.dump(output, f, indent=2)

In [3]:
def main():
    # 1. Split dataset
    coco, train_imgs, val_imgs, test_imgs = split_dataset()

    train_json = create_subset(coco, train_imgs, "train")
    val_json = create_subset(coco, val_imgs, "val")
    test_json = create_subset(coco, test_imgs, "test")

    # 2. Register
    register_dataset(train_json, val_json)

    # 3. Config
    cfg = setup_config()

    # 4. Train
    cfg = train_model(cfg)

    # 5. Predictor
    predictor = load_predictor(cfg)
    metadata = MetadataCatalog.get("bom_train")

    # 6. Run inference on test set
    print("🔍 Running inference...")

    for img in test_imgs:
        image_path = os.path.join(IMAGE_DIR, img["file_name"])

        objects = process_image(image_path, predictor, metadata)
        objects = run_ocr_stub(objects)

        save_output(image_path, objects)

    print("🎯 DONE!")

if __name__ == "__main__":
    main()

NameError: name 'split_dataset' is not defined